# 1. Limpeza dos dados: Bike Sharing (Capital Bikeshare)

**Objetivo:** ler o CSV bruto (`data/raw/hour.csv`), corrigir os problemas e salvar uma versão limpa em `data/processed/`.

Decisões tomadas aqui:
- remover o índice `instant` e renomear colunas abreviadas (`hr` → `hour`, `cnt` mantém o nome);
- criar uma coluna `datetime` (data + hora), usada para registrar cada perfil do whylogs com a data do lote (`dataset_timestamp`);
- decodificar as categorias (`season`, `weather`, `weekday`) e o ano (`0/1` → `2011/2012`);
- desfazer a normalização do clima (`temp`, `atemp`, `humidity`, `windspeed`) para unidades reais;
- marcar como ausente a umidade = 0 (falha de sensor);
- **documentar** (sem preencher) as horas que não aparecem no arquivo.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

RAW = Path("../data/raw/hour.csv")
OUT = Path("../data/processed/bike_clean.parquet")
OUT.parent.mkdir(parents=True, exist_ok=True)

## Leitura

In [2]:
df = pd.read_csv(RAW)
print(df.shape)
df.head(3)

(17379, 17)


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32


## Nomes, datas e índice

- `instant` é só um contador de linhas;
- `dteday` + `hr` viram um `datetime` único por hora;
- `yr` vem codificado como 0 (2011) e 1 (2012).

In [3]:
df = df.drop(columns=["instant"]).rename(columns={
    "dteday": "date", "yr": "year", "mnth": "month", "hr": "hour",
    "weathersit": "weather", "hum": "humidity",
})
df["date"] = pd.to_datetime(df["date"])
df["datetime"] = df["date"] + pd.to_timedelta(df["hour"], unit="h")
df["year"] = df["year"].map({0: 2011, 1: 2012})
df[["date", "datetime", "year", "month", "hour"]].head()

,date,datetime,year,month,hour
0,2011-01-01,2011-01-01 00:00:00,2011,1,0
1,2011-01-01,2011-01-01 01:00:00,2011,1,1
2,2011-01-01,2011-01-01 02:00:00,2011,1,2
3,2011-01-01,2011-01-01 03:00:00,2011,1,3
4,2011-01-01,2011-01-01 04:00:00,2011,1,4


## Categorias

O README original da UCI diz `season: 1 = spring, 2 = summer, 3 = fall, 4 = winter`, mas os dados mostram `season = 1` em janeiro e fevereiro. A codificação real segue as estações astronômicas do hemisfério norte: **1 = inverno**, 2 = primavera, 3 = verão, 4 = outono. Conferimos pelo cruzamento com o mês antes de decodificar.

In [4]:
pd.crosstab(df["month"], df["season"])

season,1,2,3,4
month,,,,
1,1429,0,0,0
2,1341,0,0,0
3,949,524,0,0
4,0,1437,0,0
5,0,1488,0,0
6,0,960,480,0
7,0,0,1488,0
8,0,0,1475,0
9,0,0,1053,384


In [5]:
df["season"] = df["season"].map({1: "winter", 2: "spring", 3: "summer", 4: "fall"})
df["weather"] = df["weather"].map({
    1: "clear", 2: "mist", 3: "light_rain_snow", 4: "heavy_rain_snow",
})
df["weekday"] = df["weekday"].map({
    0: "sun", 1: "mon", 2: "tue", 3: "wed", 4: "thu", 5: "fri", 6: "sat",
})
for c in ["season", "weather", "weekday"]:
    print(df[c].value_counts().to_dict())

{'summer': 4496, 'spring': 4409, 'winter': 4242, 'fall': 4232}
{'clear': 11413, 'mist': 4544, 'light_rain_snow': 1419, 'heavy_rain_snow': 3}
{'sat': 2512, 'sun': 2502, 'fri': 2487, 'mon': 2479, 'wed': 2475, 'thu': 2471, 'tue': 2453}


`heavy_rain_snow` tem só 3 horas no dataset inteiro. Mantemos a categoria (é real), mas ela tende a aparecer como "categoria nova" ou instável nas comparações entre períodos.

## Clima em unidades reais

Segundo a documentação da UCI, as variáveis de clima foram divididas pelo máximo: `temp` / 41 °C, `atemp` / 50 °C, `hum` / 100, `windspeed` / 67. Desfazemos isso para os relatórios ficarem legíveis (modelos de árvore não se importam com a escala).

In [6]:
df["temp"] = (df["temp"] * 41).round(2)
df["atemp"] = (df["atemp"] * 50).round(2)
df["humidity"] = (df["humidity"] * 100).round(0)
df["windspeed"] = (df["windspeed"] * 67).round(2)
df[["temp", "atemp", "humidity", "windspeed"]].describe().round(1)

,temp,atemp,humidity,windspeed
count,17379.0,17379.0,17379.0,17379.0
mean,20.4,23.8,62.7,12.7
std,7.9,8.6,19.3,8.2
min,0.8,0.0,0.0,0.0
25%,13.9,16.7,48.0,7.0
50%,20.5,24.2,63.0,13.0
75%,27.1,31.1,78.0,17.0
max,41.0,50.0,100.0,57.0


## Umidade = 0 e vento = 0

Umidade relativa de 0% não acontece em Washington: é falha de sensor. Esses valores viram `NaN` (a pipeline imputa a mediana).

Vento = 0 é bem mais frequente e pode ser calmaria real ou falta de medição; **mantemos** e só registramos.

In [7]:
print("linhas com humidity == 0:", (df["humidity"] == 0).sum())
print(df.loc[df["humidity"] == 0, "date"].dt.date.value_counts())
df.loc[df["humidity"] == 0, "humidity"] = np.nan

print("\nlinhas com windspeed == 0:", (df["windspeed"] == 0).sum(),
      f"({(df['windspeed'] == 0).mean():.1%})")

linhas com humidity == 0: 22
date
2011-03-10    22
Name: count, dtype: int64

linhas com windspeed == 0: 2180 (12.5%)


## Horas ausentes

Dois anos têm 17.544 horas, mas o arquivo tem menos linhas. Horas sem nenhum aluguel ou com o sistema fora do ar simplesmente não aparecem. **Não preenchemos** essas horas (não sabemos se o certo seria 0 ou ausente), mas é importante saber onde estão: o furacão Sandy (29–30/out/2012) derruba o sistema por quase dois dias, e isso vai aparecer no monitoramento de outubro de 2012.

In [8]:
full = pd.date_range(df["datetime"].min(), df["datetime"].max(), freq="h")
missing = full.difference(df["datetime"])
print("horas esperadas:", len(full), "| presentes:", len(df), "| ausentes:", len(missing))

miss_by_day = pd.Series(missing.date).value_counts().sort_values(ascending=False)
miss_by_day.head(8).rename("horas ausentes").to_frame()

horas esperadas: 17544 | presentes: 17379 | ausentes: 165


,horas ausentes
2012-10-29,23
2011-01-27,16
2012-10-30,13
2011-01-18,12
2011-01-26,8
2011-08-28,7
2011-08-27,6
2011-02-22,6


## Validações

In [9]:
assert (df["cnt"] == df["casual"] + df["registered"]).all(), "cnt != casual + registered"
assert df["datetime"].is_unique, "hora duplicada"
assert df["cnt"].gt(0).all(), "cnt <= 0"
assert df["hour"].between(0, 23).all()
assert df[["season", "weather", "weekday"]].notna().all().all(), "categoria não mapeada"
assert df["year"].isin([2011, 2012]).all()

print("OK. shape final:", df.shape)
print("\nLinhas por ano:")
print(df["year"].value_counts().sort_index())

OK. shape final: (17379, 17)

Linhas por ano:
year
2011    8645
2012    8734
Name: count, dtype: int64


## Salvar

`casual` e `registered` **ficam** no arquivo limpo porque ajudam a explicar o drift na EDA, mas **não podem ser usadas como features**: `cnt = casual + registered`.

In [10]:
cols = ["datetime", "date", "year", "month", "hour", "season", "weekday", "holiday", "workingday",
        "weather", "temp", "atemp", "humidity", "windspeed", "casual", "registered", "cnt"]
df = df[cols].sort_values("datetime").reset_index(drop=True)
df.to_parquet(OUT, index=False)
pd.read_parquet(OUT).dtypes

datetime      datetime64[ns]
date          datetime64[ns]
year                   int64
month                  int64
hour                   int64
season                object
weekday               object
holiday                int64
workingday             int64
weather               object
temp                 float64
atemp                float64
humidity             float64
windspeed            float64
casual                 int64
registered             int64
cnt                    int64
dtype: object